# 1.SETUP

## 1.1 LIBRARIES

In [1]:
import os
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
import requests
import zipfile
import io

## 1.2 SPARK SESSION

In [2]:
spark = SparkSession.builder.appName("SCR_Processing").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 20:38:46 WARN Utils: Your hostname, MacBook-Air-de-Mateus.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/06/23 20:38:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 20:38:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1.3 DATA READ

In [12]:
SCR_BASE_DIR = "/Volumes/SSD/SCR_DATA_RAW"
anos = [d for d in os.listdir(SCR_BASE_DIR) if os.path.isdir(os.path.join(SCR_BASE_DIR, d)) and d.isdigit()]
dfs = []

for ano in anos:
    ano_dir = os.path.join(SCR_BASE_DIR, ano)
    arquivos_csv = [os.path.join(ano_dir, f) for f in os.listdir(ano_dir) if f.endswith(".csv")]
    for arquivo in arquivos_csv:
        print(f"Lendo {arquivo}")
        df_tmp = spark.read.csv(arquivo, header=True, sep=";", inferSchema=False)
        # Conversão decimal
        df_tmp = df_tmp.withColumn('a_vencer_ate_90_dias', F.regexp_replace('a_vencer_ate_90_dias', ',', '.').cast('double')) \
                       .withColumn('a_vencer_de_91_ate_360_dias', F.regexp_replace('a_vencer_de_91_ate_360_dias', ',', '.').cast('double')) \
                       .withColumn('a_vencer_de_361_ate_1080_dias', F.regexp_replace('a_vencer_de_361_ate_1080_dias', ',', '.').cast('double')) \
                       .withColumn('a_vencer_de_1081_ate_1800_dias', F.regexp_replace('a_vencer_de_1081_ate_1800_dias', ',', '.').cast('double')) \
                       .withColumn('a_vencer_de_1801_ate_5400_dias', F.regexp_replace('a_vencer_de_1801_ate_5400_dias', ',', '.').cast('double')) \
                       .withColumn('a_vencer_acima_de_5400_dias', F.regexp_replace('a_vencer_acima_de_5400_dias', ',', '.').cast('double')) \
                       .withColumn('carteira_a_vencer', F.regexp_replace('carteira_a_vencer', ',', '.').cast('double')) \
                       .withColumn('vencido_de_15_ate_90_dias', F.regexp_replace('vencido_de_15_ate_90_dias', ',', '.').cast('double')) \
                       .withColumn('vencido_acima_de_90_dias', F.regexp_replace('vencido_acima_de_90_dias', ',', '.').cast('double')) \
                       .withColumn('carteira_vencida', F.regexp_replace('carteira_vencida', ',', '.').cast('double')) \
                       .withColumn('carteira_ativa', F.regexp_replace('carteira_ativa', ',', '.').cast('double')) \
                       .withColumn('carteira_inadimplencia', F.regexp_replace('carteira_inadimplencia', ',', '.').cast('double')) \
                       .withColumn('ativo_problematico', F.regexp_replace('ativo_problematico', ',', '.').cast('double'))
        dfs.append(df_tmp)

df = dfs[0]
for df_tmp in dfs[1:]:
    df = df.unionByName(df_tmp)


Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201304.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201310.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201311.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201305.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201307.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201306.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201312.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201302.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201303.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201301.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201308.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2013/scrdata_201309.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2014/scrdata_201401.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2014/scrdata_201402.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2014/scrdata_201403.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2014/scrdata_201407.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2014/scrdata_201406.csv
Lendo /Volumes/SSD/SCR_DATA_RAW/2014/scrdata_201

# 2. TRATAMENT

## 2.1 PORTFOLIO GROUPING

*** 

### Objetivo

O SCR.data disponibiliza informações agregadas em múltiplas dimensões, incluindo tipo de cliente, modalidade de crédito, porte, unidade da federação, segmento institucional, ocupação/CNAE, origem dos recursos e indexador.

Entretanto, a utilização simultânea de todas essas dimensões resulta em elevado nível de fragmentação dos dados, gerando séries temporais com baixa representatividade e maior volatilidade.

Dessa forma, foi necessário definir um critério de agrupamento capaz de representar carteiras com características de risco semelhantes, em linha com o conceito de agrupamento de ativos financeiros previsto pelo IFRS 9.

### Definição da Carteira

A unidade analítica adotada neste estudo é a carteira homogênea, definida pela combinação das seguintes variáveis:

\[
Carteira = Cliente + Modalidade + Porte
\]

onde:

- **Cliente**: Pessoa Física (PF) ou Pessoa Jurídica (PJ);
- **Modalidade**: categoria da operação de crédito;
- **Porte**: faixa de renda (PF) ou porte empresarial (PJ).

### Justificativa

A escolha dessas variáveis busca capturar características estruturais de risco dos tomadores e das operações de crédito, preservando simultaneamente volume suficiente de observações para análises temporais e modelagem estatística.

As demais dimensões disponíveis no SCR.data (UF, CNAE/Ocupação, Segmento Institucional, Origem dos Recursos e Indexador) não foram utilizadas na definição inicial das carteiras, pois sua inclusão aumentaria significativamente a granularidade dos dados, reduzindo a estabilidade das séries temporais e elevando a incidência de observações com baixa representatividade.

Após o agrupamento, todas as métricas de risco e deterioração passam a ser calculadas no nível da carteira homogênea.

In [16]:
df_group = df\
                .select('data_base','cliente','modalidade','porte','a_vencer_ate_90_dias','a_vencer_de_91_ate_360_dias','a_vencer_de_361_ate_1080_dias','a_vencer_de_1081_ate_1800_dias',
                        'a_vencer_de_1801_ate_5400_dias','a_vencer_acima_de_5400_dias','carteira_a_vencer','vencido_de_15_ate_90_dias','vencido_acima_de_90_dias','carteira_vencida',
                        'carteira_ativa','carteira_inadimplencia','ativo_problematico')\
                .groupby('data_base','cliente','modalidade','porte')\
                .agg(
                    F.sum('a_vencer_ate_90_dias').alias('a_vencer_ate_90_dias'),
                    F.sum('a_vencer_de_91_ate_360_dias').alias('a_vencer_de_91_ate_360_dias'),
                    F.sum('a_vencer_de_361_ate_1080_dias').alias('a_vencer_de_361_ate_1080_dias'),
                    F.sum('a_vencer_de_1081_ate_1800_dias').alias('a_vencer_de_1081_ate_1800_dias'),
                    F.sum('a_vencer_de_1801_ate_5400_dias').alias('a_vencer_de_1801_ate_5400_dias'),
                    F.sum('a_vencer_acima_de_5400_dias').alias('a_vencer_acima_de_5400_dias'),
                    F.sum('carteira_a_vencer').alias('carteira_a_vencer'),
                    F.sum('vencido_de_15_ate_90_dias').alias('vencido_de_15_ate_90_dias'),
                    F.sum('vencido_acima_de_90_dias').alias('vencido_acima_de_90_dias'),
                    F.sum('carteira_vencida').alias('carteira_vencida'),
                    F.sum('carteira_ativa').alias('carteira_ativa'),
                    F.sum('carteira_inadimplencia').alias('carteira_inadimplencia'),
                    F.sum('ativo_problematico').alias('ativo_problematico'),
                )

        

In [ ]:
df.count()

# Total de linhas sem agrupar 90.513.747


In [ ]:
df_group.count()

#Total de linhas após agrupamento 17.457

In [ ]:
df_group\
    .coalesce(1)\
    .write.mode("overwrite").option("header", True)\
    .csv("/Volumes/SSD/SCR_DATA_RAW/Carteira_Homogenea")

26/06/23 22:10:49 WARN DAGScheduler: Broadcasting large task binary with size 8.8 MiB
26/06/23 22:33:58 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
